In [1]:
import pandas as pd

files = [
    "dataset.csv",
    "nigerian_education_dropout_rates.csv",
    "student_features.csv",
    "student_target.csv",
    "student-mat.csv",
    "student-por.csv"
]

data_info = {}

for f in files:
    try:
        df = pd.read_csv(f)
        data_info[f] = {
            "columns": df.columns.tolist(),
            "shape": df.shape,
            "head": df.head(2)
        }
    except Exception as e:
        data_info[f] = str(e)

print(data_info)


{'dataset.csv': {'columns': ['Marital status', 'Application mode', 'Application order', 'Course', 'Daytime/evening attendance', 'Previous qualification', 'Nacionality', "Mother's qualification", "Father's qualification", "Mother's occupation", "Father's occupation", 'Displaced', 'Educational special needs', 'Debtor', 'Tuition fees up to date', 'Gender', 'Scholarship holder', 'Age at enrollment', 'International', 'Curricular units 1st sem (credited)', 'Curricular units 1st sem (enrolled)', 'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (approved)', 'Curricular units 1st sem (grade)', 'Curricular units 1st sem (without evaluations)', 'Curricular units 2nd sem (credited)', 'Curricular units 2nd sem (enrolled)', 'Curricular units 2nd sem (evaluations)', 'Curricular units 2nd sem (approved)', 'Curricular units 2nd sem (grade)', 'Curricular units 2nd sem (without evaluations)', 'Unemployment rate', 'Inflation rate', 'GDP', 'Target'], 'shape': (4424, 35), 'head':    Marit

In [2]:
# 1. Combine Feature and Target (Horizontal)
# This assumes they are ordered the same. Given the file names, it's highly likely.
features = pd.read_csv('student_features.csv')
target = pd.read_csv('student_target.csv')

# Check if row counts match
print(f"Features rows: {len(features)}, Target rows: {len(target)}")

# Combine columns (axis=1)
# We use concat instead of merge because there is no ID column, we assume row index alignment.
portuguese_higher_ed = pd.concat([features, target], axis=1)

# Clean duplicate columns if any (e.g. if Target was in features too)
# This removes columns with duplicate names
portuguese_higher_ed = portuguese_higher_ed.loc[:, ~portuguese_higher_ed.columns.duplicated()]

# Save
portuguese_higher_ed.to_csv('combined_higher_ed_students.csv', index=False)


# 2. Combine Math and Portuguese (Vertical & Horizontal)
mat = pd.read_csv('student-mat.csv')
por = pd.read_csv('student-por.csv')

# Vertical Stack (simply adding rows)
# Add a column to track the source subject
mat['Subject'] = 'Math'
por['Subject'] = 'Portuguese'
secondary_stacked = pd.concat([mat, por], ignore_index=True)
secondary_stacked.to_csv('stacked_secondary_students.csv', index=False)

# Horizontal Merge (Finding students who took both)
# According to UCI dataset description, these attributes identify a student uniquely
merge_cols = ['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 
              'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'nursery', 'internet']

secondary_merged = pd.merge(mat, por, on=merge_cols, suffixes=('_Math', '_Por'))
secondary_merged.to_csv('merged_secondary_common_students.csv', index=False)

print("Created: combined_higher_ed_students.csv, stacked_secondary_students.csv, merged_secondary_common_students.csv")


Features rows: 4424, Target rows: 4424
Created: combined_higher_ed_students.csv, stacked_secondary_students.csv, merged_secondary_common_students.csv


In [3]:
import pandas as pd

# Load the dataset
df = pd.read_csv('combined_higher_ed_students.csv')

# Display basic info and first few rows
print("Data Info:")
print(df.info())
print("\nUnique values in 'Course':", df['Course'].nunique())
print("\nDescriptive stats for Inflation rate and GDP:")
print(df[['Inflation rate', 'GDP']].describe())

# Check for columns with many decimal places
print("\nSample rows to check decimal precision:")
print(df.head())

Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4424 entries, 0 to 4423
Data columns (total 37 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Marital Status                                  4424 non-null   int64  
 1   Application mode                                4424 non-null   int64  
 2   Application order                               4424 non-null   int64  
 3   Course                                          4424 non-null   int64  
 4   Daytime/evening attendance                      4424 non-null   int64  
 5   Previous qualification                          4424 non-null   int64  
 6   Previous qualification (grade)                  4424 non-null   float64
 7   Nacionality                                     4424 non-null   int64  
 8   Mother's qualification                          4424 non-null   int64  
 9   Father's qualification        

In [4]:
# Define the course mapping based on the dataset's documentation
course_mapping = {
    33: 'Biofuel Production Technologies',
    171: 'Animation and Multimedia Design',
    8014: 'Social Service (evening attendance)',
    9003: 'Agronomy',
    9070: 'Communication Design',
    9085: 'Veterinary Nursing',
    9119: 'Informatics Engineering',
    9130: 'Journalism and Communication',
    9147: 'Management',
    9238: 'Social Service',
    9254: 'Tourism',
    9500: 'Nursing',
    9556: 'Oral Hygiene',
    9670: 'Management (evening attendance)',
    9773: 'Journalism and Communication',
    9853: 'Basic Education',
    9991: 'Equiniculture'
}

# Apply mapping to 'Course' column
df['Course'] = df['Course'].map(course_mapping).fillna(df['Course'])

# Fix the "too far beyond point value" by rounding all float columns to 2 decimal places
float_cols = df.select_dtypes(include=['float64']).columns
df[float_cols] = df[float_cols].round(2)

# For Inflation rate and GDP, since the user highlighted negatives as an issue,
# I will ensure they are at least rounded. 
# Economic data can be negative, but if the request implies a 'fix' for 'wrong' negatives:
# However, usually 'cleaning' means fixing outliers. 
# I will keep the values but round them to look cleaner.

# Let's check for any other 'unusually' large or precise values
print("Cleaned Data Head:")
print(df[['Course', 'Inflation rate', 'GDP', 'Curricular units 1st sem (grade)']].head())

# Save the cleaned dataset
df.to_csv('cleaned_higher_ed_students.csv', index=False)

Cleaned Data Head:
                                Course  Inflation rate   GDP  \
0      Animation and Multimedia Design             1.4  1.74   
1                              Tourism            -0.3  0.79   
2                 Communication Design             1.4  1.74   
3         Journalism and Communication            -0.8 -3.12   
4  Social Service (evening attendance)            -0.3  0.79   

   Curricular units 1st sem (grade)  
0                              0.00  
1                             14.00  
2                              0.00  
3                             13.43  
4                             12.33  


In [5]:
# Check course distribution
course_counts = df['Course'].value_counts()
print("Course Distribution:")
print(course_counts)

# Check for any remaining extremely high precision floats or negative issues
print("\nDescriptive statistics for key columns after cleaning:")
print(df[['Inflation rate', 'GDP', 'Admission grade', 'Curricular units 1st sem (grade)']].describe())

Course Distribution:
Course
Nursing                                766
Journalism and Communication           472
Management                             380
Social Service                         355
Veterinary Nursing                     337
Management (evening attendance)        268
Equiniculture                          268
Tourism                                252
Communication Design                   226
Animation and Multimedia Design        215
Social Service (evening attendance)    215
Agronomy                               210
Basic Education                        192
Informatics Engineering                170
Oral Hygiene                            86
Biofuel Production Technologies         12
Name: count, dtype: int64

Descriptive statistics for key columns after cleaning:
       Inflation rate          GDP  Admission grade  \
count     4424.000000  4424.000000      4424.000000   
mean         1.228029     0.001969       126.978119   
std          1.382711     2.269935   

In [6]:
# Final check of the actual data values
print(df[['Curricular units 1st sem (grade)', 'Curricular units 2nd sem (grade)', 'Unemployment rate', 'Inflation rate', 'GDP']].head())

   Curricular units 1st sem (grade)  Curricular units 2nd sem (grade)  \
0                              0.00                              0.00   
1                             14.00                             13.67   
2                              0.00                              0.00   
3                             13.43                             12.40   
4                             12.33                             13.00   

   Unemployment rate  Inflation rate   GDP  
0               10.8             1.4  1.74  
1               13.9            -0.3  0.79  
2               10.8             1.4  1.74  
3                9.4            -0.8 -3.12  
4               13.9            -0.3  0.79  


In [7]:
import pandas as pd

# Load the dataset
df = pd.read_csv('merged_secondary_common_students.csv')

# Display column names and head to verify
print("Columns in the dataset:")
print(df.columns.tolist())
print("\nFirst few rows:")
print(df.head())

Columns in the dataset:
['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian_Math', 'traveltime_Math', 'studytime_Math', 'failures_Math', 'schoolsup_Math', 'famsup_Math', 'paid_Math', 'activities_Math', 'nursery', 'higher_Math', 'internet', 'romantic_Math', 'famrel_Math', 'freetime_Math', 'goout_Math', 'Dalc_Math', 'Walc_Math', 'health_Math', 'absences_Math', 'G1_Math', 'G2_Math', 'G3_Math', 'Subject_Math', 'guardian_Por', 'traveltime_Por', 'studytime_Por', 'failures_Por', 'schoolsup_Por', 'famsup_Por', 'paid_Por', 'activities_Por', 'higher_Por', 'romantic_Por', 'famrel_Por', 'freetime_Por', 'goout_Por', 'Dalc_Por', 'Walc_Por', 'health_Por', 'absences_Por', 'G1_Por', 'G2_Por', 'G3_Por', 'Subject_Por']

First few rows:
  school sex  age address famsize Pstatus  Medu  Fedu     Mjob      Fjob  ...  \
0     GP   F   18       U     GT3       A     4     4  at_home   teacher  ...   
1     GP   F   17       U     GT3       T     1     1

In [8]:
cols_to_convert = [
    'schoolsup_Math', 'famsup_Math', 'paid_Math', 'activities_Math',
    'nursery', 'higher_Math', 'internet', 'romantic_Math',
    'schoolsup_Por', 'famsup_Por', 'paid_Por', 'activities_Por',
    'higher_Por', 'romantic_Por'
]

# Map 'yes' to 1 and 'no' to 0
for col in cols_to_convert:
    if col in df.columns:
        df[col] = df[col].map({'yes': 1, 'no': 0})

# Verify conversion
print(df[cols_to_convert].head())

# Save to a new CSV
output_file = 'cleaned_merged_secondary_common_students.csv'
df.to_csv(output_file, index=False)
print(f"\nSaved cleaned data to {output_file}")

   schoolsup_Math  famsup_Math  paid_Math  activities_Math  nursery  \
0               1            0          0                0        1   
1               0            1          0                0        0   
2               1            0          1                0        1   
3               0            1          1                1        1   
4               0            1          1                0        1   

   higher_Math  internet  romantic_Math  schoolsup_Por  famsup_Por  paid_Por  \
0            1         0              0              1           0         0   
1            1         1              0              0           1         0   
2            1         1              0              1           0         0   
3            1         1              1              0           1         0   
4            1         0              0              0           1         0   

   activities_Por  higher_Por  romantic_Por  
0               0           1             0  
